# IEMOCAP Dataset Analyzer
Systematically explores the structure of `iemocap_multi_features.pkl` to decode what every field, number, and array represents — and how they feed into GraphSmile.

In [1]:
import pickle, numpy as np, pprint

PKL_PATH = "Dataset/CFN-ESA/iemocap_multi_features.pkl"

data = pickle.load(open(PKL_PATH, "rb"), encoding="latin1")

print(f"Type of loaded object : {type(data)}")
print(f"Number of top-level fields : {len(data)}")
for i, item in enumerate(data):
    t = type(item).__name__
    sz = len(item) if hasattr(item, '__len__') else "scalar"
    print(f"  [{i}] type={t:10s}  len={sz}")

Type of loaded object : <class 'list'>
Number of top-level fields : 12
  [0] type=dict        len=151
  [1] type=dict        len=151
  [2] type=dict        len=151
  [3] type=dict        len=151
  [4] type=dict        len=151
  [5] type=dict        len=151
  [6] type=dict        len=151
  [7] type=dict        len=151
  [8] type=dict        len=151
  [9] type=dict        len=151
  [10] type=list        len=120
  [11] type=list        len=31


## Field Names & Assignment
The pkl is a tuple of 12 items (matching the `IEMOCAPDataset_BERT` constructor).  
Below we unpack and name each one.

In [2]:
(
    videoIDs,
    videoSpeakers,
    videoLabels,
    videoText0,
    videoText1,
    videoText2,
    videoText3,
    videoAudio,
    videoVisual,
    videoSentence,
    trainVid,
    testVid,
) = data

# Show all dialogue IDs
all_vids = sorted(videoIDs.keys())
print(f"Total dialogues  : {len(all_vids)}")
print(f"Train dialogues  : {len(trainVid)}")
print(f"Test  dialogues  : {len(testVid)}")
print(f"\nSample dialogue IDs (first 10): {all_vids[:10]}")

Total dialogues  : 151
Train dialogues  : 120
Test  dialogues  : 31

Sample dialogue IDs (first 10): ['Ses01F_impro01', 'Ses01F_impro02', 'Ses01F_impro03', 'Ses01F_impro04', 'Ses01F_impro05', 'Ses01F_impro06', 'Ses01F_impro07', 'Ses01F_script01_1', 'Ses01F_script01_2', 'Ses01F_script01_3']


## 1. `videoIDs` — utterance IDs per dialogue

In [3]:
sample_vid = all_vids[0]
print(f"Sample dialogue : '{sample_vid}'")
print(f"  Utterance IDs : {videoIDs[sample_vid]}")
print(f"  # utterances  : {len(videoIDs[sample_vid])}")
print()
# Distribution of dialogue lengths
lengths = [len(videoIDs[v]) for v in all_vids]
print(f"Dialogue length stats:")
print(f"  min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}, median={np.median(lengths):.0f}")

Sample dialogue : 'Ses01F_impro01'
  Utterance IDs : ['Ses01F_impro01_F000', 'Ses01F_impro01_M000', 'Ses01F_impro01_F001', 'Ses01F_impro01_M001', 'Ses01F_impro01_F002', 'Ses01F_impro01_M002', 'Ses01F_impro01_M003', 'Ses01F_impro01_F005', 'Ses01F_impro01_M004', 'Ses01F_impro01_M005', 'Ses01F_impro01_F006', 'Ses01F_impro01_M006', 'Ses01F_impro01_F007', 'Ses01F_impro01_M007', 'Ses01F_impro01_F008', 'Ses01F_impro01_M008', 'Ses01F_impro01_F009', 'Ses01F_impro01_M009', 'Ses01F_impro01_F011', 'Ses01F_impro01_M010', 'Ses01F_impro01_F012', 'Ses01F_impro01_M011', 'Ses01F_impro01_F013', 'Ses01F_impro01_F014', 'Ses01F_impro01_M013', 'Ses01F_impro01_F015']
  # utterances  : 26

Dialogue length stats:
  min=8, max=110, mean=49.2, median=47


## 2. `videoSpeakers` — who speaks each utterance
Each utterance is labeled `'M'` (Male) or `'F'` (Female).  
In `__getitem__` this becomes a one-hot vector: `M → [1,0]`, `F → [0,1]`.

In [4]:
print(f"Raw speaker labels for '{sample_vid}': {videoSpeakers[sample_vid]}")
print()
# Count speaker turns
from collections import Counter
total_m, total_f = 0, 0
for v in all_vids:
    c = Counter(videoSpeakers[v])
    total_m += c.get('M', 0)
    total_f += c.get('F', 0)
print(f"Total M utterances : {total_m}")
print(f"Total F utterances : {total_f}")
print(f"→ In the model: M→[1,0] and F→[0,1] (speaker identity vector, shape [seq_len, 2])")

Raw speaker labels for 'Ses01F_impro01': ['F', 'M', 'F', 'M', 'F', 'M', 'M', 'F', 'M', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'F', 'M', 'F']

Total M utterances : 3939
Total F utterances : 3494
→ In the model: M→[1,0] and F→[0,1] (speaker identity vector, shape [seq_len, 2])


## 3. `videoLabels` — emotion class per utterance
IEMOCAP-6 label mapping (used by `IEMOCAPDataset_BERT`):

| Index | Emotion      | Sentiment (derived)  |
|-------|-------------|----------------------|
| 0     | neutral     | positive (→ 2)       |
| 1     | frustration | negative (→ 0)       |
| 2     | sadness     | neutral  (→ 1)       |
| 3     | anger       | negative (→ 0)       |
| 4     | happiness   | positive (→ 2)       |
| 5     | excitement  | negative (→ 0)       |

Sentiment derivation rule in `IEMOCAPDataset_BERT.__init__`:
```python
{1, 3, 5}  → 0  (negative)   # frustration, anger, excitement
 2         → 1  (neutral)    # sadness
{0, 4}     → 2  (positive)   # neutral, happiness
```
Note: "excitement" is treated as negative valence here per the IEMOCAP annotation convention.  
"sadness" maps to neutral (not deeply negative) in this scheme.

In [5]:
EMOTION_MAP = {0: 'neutral', 1: 'frustration', 2: 'sadness', 3: 'anger', 4: 'happiness', 5: 'excitement'}
SENTIMENT_FROM_LABEL = {1: 'neg(0)', 3: 'neg(0)', 5: 'neg(0)', 2: 'neu(1)', 0: 'pos(2)', 4: 'pos(2)'}

raw = videoLabels[sample_vid]
print(f"Emotion labels for '{sample_vid}':")
for i, (utt_id, lbl) in enumerate(zip(videoIDs[sample_vid], raw)):
    print(f"  utt {i:2d}  id={utt_id}  label={lbl}  emotion={EMOTION_MAP[lbl]:12s}  sentiment={SENTIMENT_FROM_LABEL[lbl]}")

print()
all_labels = [l for v in all_vids for l in videoLabels[v]]
print(f"Total utterances : {len(all_labels)}")
emo_counts = Counter(all_labels)
print("\nEmotion class distribution (across full dataset):")
for k in sorted(emo_counts):
    print(f"  {k} ({EMOTION_MAP[k]:12s}) : {emo_counts[k]:4d}  ({100*emo_counts[k]/len(all_labels):.1f}%)")

Emotion labels for 'Ses01F_impro01':
  utt  0  id=Ses01F_impro01_F000  label=2  emotion=sadness       sentiment=neu(1)
  utt  1  id=Ses01F_impro01_M000  label=5  emotion=excitement    sentiment=neg(0)
  utt  2  id=Ses01F_impro01_F001  label=2  emotion=sadness       sentiment=neu(1)
  utt  3  id=Ses01F_impro01_M001  label=5  emotion=excitement    sentiment=neg(0)
  utt  4  id=Ses01F_impro01_F002  label=2  emotion=sadness       sentiment=neu(1)
  utt  5  id=Ses01F_impro01_M002  label=5  emotion=excitement    sentiment=neg(0)
  utt  6  id=Ses01F_impro01_M003  label=5  emotion=excitement    sentiment=neg(0)
  utt  7  id=Ses01F_impro01_F005  label=2  emotion=sadness       sentiment=neu(1)
  utt  8  id=Ses01F_impro01_M004  label=5  emotion=excitement    sentiment=neg(0)
  utt  9  id=Ses01F_impro01_M005  label=5  emotion=excitement    sentiment=neg(0)
  utt 10  id=Ses01F_impro01_F006  label=5  emotion=excitement    sentiment=neg(0)
  utt 11  id=Ses01F_impro01_M006  label=5  emotion=excitement

## 4. Text Features — `videoText0/1/2/3`
Four variants of text embeddings (different BERT layers/pooling).  
Each is a list of utterance vectors of shape `[seq_len, text_dim]` per dialogue.  
In the model, `embedding_dims[0] = 1024` for IEMOCAP.

In [6]:
for name, field in [("videoText0", videoText0), ("videoText1", videoText1),
                    ("videoText2", videoText2), ("videoText3", videoText3)]:
    arr = np.array(field[sample_vid])
    print(f"{name}['{sample_vid}']  shape={arr.shape}  dtype={arr.dtype}  "
          f"range=[{arr.min():.3f}, {arr.max():.3f}]  mean={arr.mean():.4f}")

print()
print("→ Each row = one utterance's text embedding (1024-dim BERT feature)")
print("→ The model projects these to hidden_dim via dim_layer_t")

videoText0['Ses01F_impro01']  shape=(26, 1024)  dtype=float32  range=[-17.064, 15.238]  mean=-0.0312
videoText1['Ses01F_impro01']  shape=(26, 1024)  dtype=float32  range=[-28.477, 3.508]  mean=-0.0308
videoText2['Ses01F_impro01']  shape=(26, 1024)  dtype=float32  range=[-31.518, 2.314]  mean=-0.0416
videoText3['Ses01F_impro01']  shape=(26, 1024)  dtype=float32  range=[-31.079, 1.599]  mean=-0.0332

→ Each row = one utterance's text embedding (1024-dim BERT feature)
→ The model projects these to hidden_dim via dim_layer_t


## 5. Visual Features — `videoVisual`
Facial/video features per utterance. `embedding_dims[1] = 342` for IEMOCAP.

In [7]:
arr_v = np.array(videoVisual[sample_vid])
print(f"videoVisual['{sample_vid}']  shape={arr_v.shape}  dtype={arr_v.dtype}")
print(f"  value range : [{arr_v.min():.4f}, {arr_v.max():.4f}]")
print(f"  mean        : {arr_v.mean():.6f}")
print(f"  std         : {arr_v.std():.6f}")
print()
print("→ Each row = one utterance's visual/facial feature vector (342-dim)")
print("→ Used as input to hetergconv_tv and hetergconv_va in the graph")

videoVisual['Ses01F_impro01']  shape=(26, 342)  dtype=float64
  value range : [0.0000, 2.5637]
  mean        : 0.103765
  std         : 0.289107

→ Each row = one utterance's visual/facial feature vector (342-dim)
→ Used as input to hetergconv_tv and hetergconv_va in the graph


## 6. Audio Features — `videoAudio`
Acoustic features per utterance. `embedding_dims[2] = 1582` for IEMOCAP.

In [8]:
arr_a = np.array(videoAudio[sample_vid])
print(f"videoAudio['{sample_vid}']  shape={arr_a.shape}  dtype={arr_a.dtype}")
print(f"  value range : [{arr_a.min():.4f}, {arr_a.max():.4f}]")
print(f"  mean        : {arr_a.mean():.6f}")
print(f"  std         : {arr_a.std():.6f}")
print()
print("→ Each row = one utterance's acoustic feature vector (1582-dim)")
print("→ Used as input to hetergconv_ta and hetergconv_va in the graph")

print()
print("=" * 60)
print("SUMMARY — Feature dimensions for IEMOCAP")
print("=" * 60)
print(f"  Text (BERT)    : {np.array(videoText0[sample_vid]).shape[1]}-dim  (4 variants: text0-3)")
print(f"  Visual         : {arr_v.shape[1]}-dim")
print(f"  Audio          : {arr_a.shape[1]}-dim")

videoAudio['Ses01F_impro01']  shape=(26, 1582)  dtype=float32
  value range : [-5.7967, 15.5634]
  mean        : 0.110854
  std         : 0.933292

→ Each row = one utterance's acoustic feature vector (1582-dim)
→ Used as input to hetergconv_ta and hetergconv_va in the graph

SUMMARY — Feature dimensions for IEMOCAP
  Text (BERT)    : 1024-dim  (4 variants: text0-3)
  Visual         : 342-dim
  Audio          : 1582-dim


## 7. `videoSentence` — raw transcription strings
The actual text of each utterance (used for reference, not as a model input directly).

In [9]:
print(f"Sentences in '{sample_vid}':")
for i, (utt_id, spk, lbl, sent) in enumerate(zip(
        videoIDs[sample_vid], videoSpeakers[sample_vid],
        videoLabels[sample_vid], videoSentence[sample_vid])):
    print(f"  [{i:2d}] id={utt_id}  {spk}  label={lbl}({EMOTION_MAP[lbl]:12s})  → \"{sent}\"")

Sentences in 'Ses01F_impro01':
  [ 0] id=Ses01F_impro01_F000  F  label=2(sadness     )  → "Excuse me."
  [ 1] id=Ses01F_impro01_M000  M  label=5(excitement  )  → "Do you have your forms?"
  [ 2] id=Ses01F_impro01_F001  F  label=2(sadness     )  → "Yeah."
  [ 3] id=Ses01F_impro01_M001  M  label=5(excitement  )  → "Let me see them."
  [ 4] id=Ses01F_impro01_F002  F  label=2(sadness     )  → "Is there a problem?"
  [ 5] id=Ses01F_impro01_M002  M  label=5(excitement  )  → "Who told you to get in this line?"
  [ 6] id=Ses01F_impro01_M003  M  label=5(excitement  )  → "Okay. But I didn't tell you to get in this line if you are filling out this particular form."
  [ 7] id=Ses01F_impro01_F005  F  label=2(sadness     )  → "Well what's the problem?  Let me change it."
  [ 8] id=Ses01F_impro01_M004  M  label=5(excitement  )  → "This form is a Z.X.four."
  [ 9] id=Ses01F_impro01_M005  M  label=5(excitement  )  → "You can't--  This is not the line for Z.X.four.  If you're going to fill out the Z.X.f

## 8. `trainVid` / `testVid` — train-test split
These are lists of dialogue IDs that determine which conversations go to training vs. test.

In [10]:
train_utts = sum(len(videoIDs[v]) for v in trainVid)
test_utts  = sum(len(videoIDs[v]) for v in testVid)

print(f"trainVid : {len(trainVid):3d} dialogues  → {train_utts} utterances")
print(f"testVid  : {len(testVid):3d} dialogues  → {test_utts}  utterances")
print()
print(f"First 5 train dialogue IDs : {list(trainVid)[:5]}")
print(f"First 5 test  dialogue IDs : {list(testVid)[:5]}")
print()

# Per-split emotion distribution
for split_name, split_vids in [("TRAIN", trainVid), ("TEST", testVid)]:
    labels = [l for v in split_vids for l in videoLabels[v]]
    c = Counter(labels)
    print(f"\n{split_name} emotion distribution:")
    for k in sorted(c):
        print(f"  {k} ({EMOTION_MAP[k]:12s}) : {c[k]:4d}  ({100*c[k]/len(labels):.1f}%)")

trainVid : 120 dialogues  → 5810 utterances
testVid  :  31 dialogues  → 1623  utterances

First 5 train dialogue IDs : ['Ses02F_script01_3', 'Ses03F_script03_1', 'Ses01M_impro03', 'Ses02M_impro07', 'Ses02M_impro08']
First 5 test  dialogue IDs : ['Ses05F_impro08', 'Ses05M_impro02', 'Ses05F_impro01', 'Ses05F_script01_2', 'Ses05F_impro05']


TRAIN emotion distribution:
  0 (neutral     ) :  504  (8.7%)
  1 (frustration ) :  839  (14.4%)
  2 (sadness     ) : 1324  (22.8%)
  3 (anger       ) :  933  (16.1%)
  4 (happiness   ) :  742  (12.8%)
  5 (excitement  ) : 1468  (25.3%)

TEST emotion distribution:
  0 (neutral     ) :  144  (8.9%)
  1 (frustration ) :  245  (15.1%)
  2 (sadness     ) :  384  (23.7%)
  3 (anger       ) :  170  (10.5%)
  4 (happiness   ) :  299  (18.4%)
  5 (excitement  ) :  381  (23.5%)


## 9. Derived Labels: Sentiment & Sentiment Shift
The pkl only stores raw emotion labels. Sentiment (3-class) and shift (binary) are computed on-the-fly by the dataloader.

In [11]:
SENTIMENT_MAP = {0: 'negative', 1: 'neutral', 2: 'positive'}

def emo_to_sentiment_6class(e):
    """IEMOCAPDataset_BERT mapping (6-class IEMOCAP)"""
    if e in [1, 3, 5]: return 0   # negative
    elif e == 2:        return 1   # neutral
    elif e in [0, 4]:   return 2   # positive

print(f"Sentiment sequence for '{sample_vid}':")
emo_seq = videoLabels[sample_vid]
sen_seq = [emo_to_sentiment_6class(e) for e in emo_seq]
for i, (e, s) in enumerate(zip(emo_seq, sen_seq)):
    print(f"  [{i:2d}] emo={e}({EMOTION_MAP[e]:12s})  → sentiment={s}({SENTIMENT_MAP[s]})")

print()
print("Sentiment shift (binary) — computed from consecutive sentiment changes:")
for i in range(1, len(sen_seq)):
    shift = 1 if sen_seq[i] != sen_seq[i-1] else 0
    print(f"  utt {i-1}→{i}  sentiment {sen_seq[i-1]}→{sen_seq[i]}  shift={shift}")

print()
print("NOTE: The SentimentShift module in module.py computes this using a windowed")
print("      comparison across 'shift_win' utterances, not just adjacent pairs.")

Sentiment sequence for 'Ses01F_impro01':
  [ 0] emo=2(sadness     )  → sentiment=1(neutral)
  [ 1] emo=5(excitement  )  → sentiment=0(negative)
  [ 2] emo=2(sadness     )  → sentiment=1(neutral)
  [ 3] emo=5(excitement  )  → sentiment=0(negative)
  [ 4] emo=2(sadness     )  → sentiment=1(neutral)
  [ 5] emo=5(excitement  )  → sentiment=0(negative)
  [ 6] emo=5(excitement  )  → sentiment=0(negative)
  [ 7] emo=2(sadness     )  → sentiment=1(neutral)
  [ 8] emo=5(excitement  )  → sentiment=0(negative)
  [ 9] emo=5(excitement  )  → sentiment=0(negative)
  [10] emo=5(excitement  )  → sentiment=0(negative)
  [11] emo=5(excitement  )  → sentiment=0(negative)
  [12] emo=5(excitement  )  → sentiment=0(negative)
  [13] emo=5(excitement  )  → sentiment=0(negative)
  [14] emo=5(excitement  )  → sentiment=0(negative)
  [15] emo=5(excitement  )  → sentiment=0(negative)
  [16] emo=5(excitement  )  → sentiment=0(negative)
  [17] emo=5(excitement  )  → sentiment=0(negative)
  [18] emo=5(excitement  ) 

## 10. Complete Data Flow: PKL → Model

```
pkl file
├── videoIDs      dict[vid → list[utt_id]]          # utterance names
├── videoSpeakers dict[vid → list['M'|'F']]          # speaker identity
├── videoLabels   dict[vid → list[int 0-5]]          # emotion class (ground truth)
├── videoText0-3  dict[vid → list[float32 (1024,)]]  # BERT text features (4 variants)
├── videoAudio    dict[vid → list[float32 (1582,)]]  # acoustic features
├── videoVisual   dict[vid → list[float32 (342,)]]   # facial/video features
├── videoSentence dict[vid → list[str]]              # raw transcriptions
├── trainVid      list[vid]                          # train dialogue IDs
└── testVid       list[vid]                          # test dialogue IDs

IEMOCAPDataset_BERT.__getitem__(index)  →  returns per dialogue:
  [0]  text0        FloatTensor [seq_len, 1024]   BERT variant 0
  [1]  text1        FloatTensor [seq_len, 1024]   BERT variant 1
  [2]  text2        FloatTensor [seq_len, 1024]   BERT variant 2
  [3]  text3        FloatTensor [seq_len, 1024]   BERT variant 3
  [4]  visual       FloatTensor [seq_len, 342]    facial features
  [5]  audio        FloatTensor [seq_len, 1582]   acoustic features
  [6]  speakers     FloatTensor [seq_len, 2]      one-hot M/F
  [7]  mask         FloatTensor [seq_len]         all-ones (for padding)
  [8]  labels_emo   LongTensor  [seq_len]         emotion class 0-5
  [9]  labels_sen   LongTensor  [seq_len]         sentiment 0=neg,1=neu,2=pos
  [10] vid          str                           dialogue ID

After collate_fn (pad_sequence):
  [0-6]  →  [max_seq_len, batch, feature_dim]  (padded across batch)
  [7-9]  →  [max_seq_len, batch]               (padded labels/mask)
  [10]   →  list of vid strings
```